### Wallmart Sales data analysis

In [30]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

In [32]:
host = 'localhost'
port = 3306
password = ''
username = 'root'
database = 'data_analysis'

In [34]:
conn = create_engine(f'mysql+pymysql://{username}:{password}@{host}:{port}/{database}')

In [7]:
walmart_data = pd.read_csv("../dataset/Walmart.csv")

In [8]:
walmart_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10051 entries, 0 to 10050
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   invoice_id      10051 non-null  int64  
 1   Branch          10051 non-null  object 
 2   City            10051 non-null  object 
 3   category        10051 non-null  object 
 4   unit_price      10020 non-null  object 
 5   quantity        10020 non-null  float64
 6   date            10051 non-null  object 
 7   time            10051 non-null  object 
 8   payment_method  10051 non-null  object 
 9   rating          10051 non-null  float64
 10  profit_margin   10051 non-null  float64
dtypes: float64(3), int64(1), object(7)
memory usage: 863.9+ KB


In [10]:
walmart_data.isnull().sum()

invoice_id         0
Branch             0
City               0
category           0
unit_price        31
quantity          31
date               0
time               0
payment_method     0
rating             0
profit_margin      0
dtype: int64

In [17]:
walmart_data['unit_price'] = walmart_data['unit_price'].str.replace("$", "")
walmart_data['unit_price'] = walmart_data['unit_price'].astype(float)
walmart_data['unit_price'].fillna(walmart_data['unit_price'].mean(), inplace=True)

/var/folders/g2/c4ppjm_x53lfmhpz_sqp533r0000gn/T/ipykernel_98606/2824543593.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  walmart_data['unit_price'].fillna(walmart_data['unit_price'].mean(), inplace=True)


In [21]:
walmart_data.isnull().sum()

invoice_id        0
Branch            0
City              0
category          0
unit_price        0
quantity          0
date              0
time              0
payment_method    0
rating            0
profit_margin     0
dtype: int64

In [20]:
walmart_data['quantity'].fillna(walmart_data['quantity'].mean(), inplace=True)

/var/folders/g2/c4ppjm_x53lfmhpz_sqp533r0000gn/T/ipykernel_98606/3086598607.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  walmart_data['quantity'].fillna(walmart_data['quantity'].mean(), inplace=True)


In [28]:
walmart_data.columns

Index(['invoice_id', 'Branch', 'City', 'category', 'unit_price', 'quantity',
       'date', 'time', 'payment_method', 'rating', 'profit_margin', 'total'],
      dtype='object')

In [27]:
walmart_data['total'] = walmart_data['quantity'] * walmart_data['unit_price']

In [29]:
walmart_data

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin,total
0,1,WALM003,San Antonio,Health and beauty,74.69,7.0,05/01/19,13:08:00,Ewallet,9.1,0.48,522.83
1,2,WALM048,Harlingen,Electronic accessories,15.28,5.0,08/03/19,10:29:00,Cash,9.6,0.48,76.40
2,3,WALM067,Haltom City,Home and lifestyle,46.33,7.0,03/03/19,13:23:00,Credit card,7.4,0.33,324.31
3,4,WALM064,Bedford,Health and beauty,58.22,8.0,27/01/19,20:33:00,Ewallet,8.4,0.33,465.76
4,5,WALM013,Irving,Sports and travel,86.31,7.0,08/02/19,10:37:00,Ewallet,5.3,0.48,604.17
...,...,...,...,...,...,...,...,...,...,...,...,...
10046,9996,WALM056,Rowlett,Fashion accessories,37.00,3.0,03/08/23,10:10:00,Cash,3.0,0.33,111.00
10047,9997,WALM030,Richardson,Home and lifestyle,58.00,2.0,22/02/21,14:20:00,Cash,7.0,0.48,116.00
10048,9998,WALM050,Victoria,Fashion accessories,52.00,3.0,15/06/23,16:00:00,Credit card,4.0,0.48,156.00
10049,9999,WALM032,Tyler,Home and lifestyle,79.00,2.0,25/02/21,12:25:00,Cash,7.0,0.48,158.00


In [36]:
walmart_data.to_sql(name="walmart", con=conn, if_exists="append", index=False)

10051

### Retrieve all unique product categories.

In [43]:
pd.read_sql("select DISTINCT category from walmart;", con=conn)

,category
0,Health and beauty
1,Electronic accessories
2,Home and lifestyle
3,Sports and travel
4,Food and beverages
5,Fashion accessories


### Find the total number of transactions per Branch.

In [46]:
pd.read_sql("select branch, count(invoice_id) as count from walmart GROUP BY Branch;", con=conn)

,branch,count
0,WALM003,187
1,WALM048,80
2,WALM067,74
3,WALM064,64
4,WALM013,57
...,...,...
95,WALM032,171
96,WALM082,187
97,WALM006,71
98,WALM092,51


### Show the top 5 transactions with the highest quantity purchased.

In [51]:
pd.read_sql("select * from walmart ORDER BY quantity DESC LIMIT 5;", con=conn)

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin,total
0,411,WALM011,Lubbock,Health and beauty,34.21,10.0,02/01/19,13:00:00,Cash,5.1,0.48,342.1
1,394,WALM099,Weatherford,Sports and travel,52.26,10.0,09/03/19,12:45:00,Credit card,6.2,0.18,522.6
2,392,WALM002,Dallas,Fashion accessories,37.95,10.0,26/01/19,14:51:00,Cash,9.7,0.36,379.5
3,388,WALM034,College Station,Health and beauty,32.32,10.0,20/02/19,16:49:00,Credit card,10.0,0.48,323.2
4,423,WALM022,Mesquite,Fashion accessories,97.21,10.0,08/02/19,13:00:00,Credit card,8.7,0.48,972.1


### Count how many transactions used Credit card as the payment method.

In [55]:
pd.read_sql("select payment_method, COUNT(payment_method) as total from walmart WHERE payment_method = 'Credit card';", con=conn)

,payment_method,total
0,Credit card,4260


### List distinct cities where Walmart branches are located.

In [58]:
pd.read_sql("Select DISTINCT City from walmart;", con=conn)

,City
0,San Antonio
1,Harlingen
2,Haltom City
3,Bedford
4,Irving
...,...
93,Laredo
94,Tyler
95,El Paso
96,Lake Jackson
